- Эта тетрадь предназначена для вычисления gram_structure и косинусной близости

**Вопросы:** 
1. Стоит ли вычислять грамматическую структуру отдельно или заведомо хранить ее в бд.;
2. Вычислять грамматическую структуру предложения или концепта (что лучше);
3. Вычислять косинусную близость предложения или концепта.

### Константы

In [11]:
# импорты
import pandas as pd
from openpyxl import load_workbook
from tqdm import tqdm
tqdm.pandas()


# файл
# FILENAME = '../results/Разметка_сравнение_RU_EN.xlsx'
FILENAME = '../results/примеры.xlsx'

# 
THRESHOLD_EQUIV   = 0.90   # ≥0.90 → equivalent по умолчанию
THRESHOLD_SHIFT   = 0.75   # 0.75–0.89 → shift

### Этап I - обработка и вычленение gram_structure

#### Грамматическая структура: подготовка

In [12]:
import spacy, stanza
# import openpyxl
# from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

In [13]:
# print("Загрузка spaCy en_core_web_trf ...")  поменяла
nlp_en = spacy.load("en_core_web_sm")

print("Загрузка Stanza RU (ru_syntagrus) ...")
nlp_ru = stanza.Pipeline(
    lang="ru",
    processors="tokenize,pos,lemma,depparse",
    tokenize_no_ssplit=True,  # весь concept_unit = одно предложение
    verbose=False,
)
print("✓ Обе модели готовы")

Загрузка Stanza RU (ru_syntagrus) ...
✓ Обе модели готовы


#### функции

In [14]:
def _fmt_deprel(label: str) -> str:
    """Нормализует deprel: сохраняет подтип только для значимых меток."""
    keep = {"nsubj", "obj", "obl", "nmod", "acl", "advcl", "csubj"}
    parts = label.split(":")
    return label if parts[0] in keep and len(parts) > 1 else parts[0]


In [15]:

def build_ud_stanza(text: str, max_depth: int = 2) -> str:
    """
    Парсит русский текст через Stanza (ru_syntagrus).

    Возвращает строку вида:
      ROOT[UPOS](лемма) + deprel(форма/UPOS) + ...
    Вложенные зависимые (глубина ≤ max_depth):
      deprel(форма/UPOS > [inner_dep(форма/UPOS), ...])
    """
    if not text or not str(text).strip():
        return ""
    text = str(text).strip().rstrip(".")
    doc = nlp_ru(text)
    if not doc.sentences:
        return text

    words = doc.sentences[0].words
    children = {w.id: [] for w in words}
    root_word = None

    for w in words:
        if w.deprel and w.deprel.lower() == "root":
            root_word = w
        if w.head and w.head in children and w.head != w.id:
            children[w.head].append(w)

    if root_word is None:
        root_word = words[0]

    def subtree(w, depth=0):
        if (w.upos or "X") == "PUNCT":
            return None
        kids = [c for c in sorted(children.get(w.id, []), key=lambda x: x.id)
                if (c.upos or "X") != "PUNCT"]
        parts = []
        for kid in kids:
            rel = _fmt_deprel(kid.deprel or "dep")
            grandkids = [gc for gc in sorted(children.get(kid.id, []), key=lambda x: x.id)
                         if (gc.upos or "X") != "PUNCT"]
            if grandkids and depth < max_depth:
                inner = ", ".join(f"{_fmt_deprel(gc.deprel or 'dep')}({gc.text}/{gc.upos})"
                                  for gc in grandkids)
                parts.append(f"{rel}({kid.text}/{kid.upos} > [{inner}])")
            else:
                parts.append(f"{rel}({kid.text}/{kid.upos})")
        label = "ROOT" if w == root_word else _fmt_deprel(w.deprel or "dep").upper()
        base = f"{label}[{w.upos or 'X'}]({w.lemma or w.text})"
        return (base + " + " + " + ".join(parts)) if parts else base

    return subtree(root_word) or text

In [16]:
def build_ud_spacy(text: str, max_depth: int = 2) -> str:
    """
    Парсит английский текст через spaCy (en_core_web_trf).
    Нотация идентична build_ud_stanza.
    """
    if not text or not str(text).strip():
        return ""
    text = str(text).strip().rstrip(".")
    doc = nlp_en(text)

    children = {t.i: [] for t in doc}
    root_tok = None
    for t in doc:
        if t.dep_ == "ROOT":
            root_tok = t
        if t.dep_ != "ROOT" and t.head.i != t.i:
            children[t.head.i].append(t)
    if root_tok is None:
        root_tok = doc[0]

    def subtree(t, depth=0):
        if t.pos_ in ("PUNCT", "SPACE"):
            return None
        kids = [c for c in sorted(children.get(t.i, []), key=lambda x: x.i)
                if c.pos_ not in ("PUNCT", "SPACE")]
        parts = []
        for kid in kids:
            rel = _fmt_deprel(kid.dep_)
            grandkids = [gc for gc in sorted(children.get(kid.i, []), key=lambda x: x.i)
                         if gc.pos_ not in ("PUNCT", "SPACE")]
            if grandkids and depth < max_depth:
                inner = ", ".join(f"{_fmt_deprel(gc.dep_)}({gc.text}/{gc.pos_})"
                                  for gc in grandkids)
                parts.append(f"{rel}({kid.text}/{kid.pos_} > [{inner}])")
            else:
                parts.append(f"{rel}({kid.text}/{kid.pos_})")
        label = "ROOT" if t == root_tok else _fmt_deprel(t.dep_).upper()
        base = f"{label}[{t.pos_}]({t.lemma_})"
        return (base + " + " + " + ".join(parts)) if parts else base

    return subtree(root_tok) or text




In [17]:
# ── Тест на примерах из корпуса ──────────────────────────────────────────
tests = [
    ("RU", "запах гари"),
    ("RU", "невыносимый смрад"),
    ("RU", "грузовик дохнул раскаленной вонью"),
    ("RU", "не вытравил из себя сладко-смердящего матушкина духа"),
    ("RU", "зефир, шумящий древесами, веет нам благоуханием, собранным со цветов"),
    ("EN", "smell of burning"),
    ("EN", "unbearable stench"),
    ("EN", "truck breathed scorching stench"),
    ("EN", "not purged the sweetly festering spirit"),
    ("EN", "zephyr wafts fragrance gathered from the flowers"),
]
print("=" * 65)
for lang, phrase in tests:
    fn = build_ud_stanza if lang == "RU" else build_ud_spacy
    print(f"\n[{lang}] «{phrase}»")
    print(f"  → {fn(phrase)}")



[RU] «запах гари»
  → ROOT[NOUN](запах) + nmod(гари/NOUN)

[RU] «невыносимый смрад»
  → ROOT[NOUN](смрад) + amod(невыносимый/ADJ)

[RU] «грузовик дохнул раскаленной вонью»
  → ROOT[VERB](дохнуть) + nsubj(грузовик/NOUN) + obl(вонью/NOUN > [amod(раскаленной/VERB)])

[RU] «не вытравил из себя сладко-смердящего матушкина духа»
  → ROOT[VERB](вытравить) + advmod(не/PART) + obl(себя/PRON > [case(из/ADP)]) + nsubj(матушкина/NOUN > [acl(смердящего/VERB), nmod(духа/NOUN)])

[RU] «зефир, шумящий древесами, веет нам благоуханием, собранным со цветов»
  → ROOT[VERB](веять) + nsubj(зефир/NOUN > [acl(шумящий/VERB)]) + iobj(нам/PRON) + obl(благоуханием/NOUN > [acl(собранным/VERB)])

[EN] «smell of burning»
  → ROOT[NOUN](smell) + prep(of/ADP > [pcomp(burning/VERB)])

[EN] «unbearable stench»
  → ROOT[NOUN](stench) + amod(unbearable/ADJ)

[EN] «truck breathed scorching stench»
  → ROOT[VERB](breathe) + nsubj(truck/NOUN) + dobj(stench/NOUN > [amod(scorching/VERB)])

[EN] «not purged the sweetly fester

#### Обработка и вычленение gram_structure

In [ ]:
df = pd.read_excel(FILENAME, sheet_name='Разметка', header=0)

# 2. Проверяем, какие колонки есть
print("Доступные колонки:", list(df.columns))

# Обрабатываем русские фразы (используем правильное имя колонки)
df['gram_structure_RU'] = df['concept_unit_RU'].apply(
    lambda x: build_ud_stanza(str(x)) if pd.notna(x) else ''
)

# Обрабатываем английские фразы
df['gram_structure_EN'] = df['concept_unit_EN'].apply(
    lambda x: build_ud_spacy(str(x)) if pd.notna(x) else ''
)

print("✅ Обработка завершена")
# print(df[['concept_unit_RU', 'gram_structure_RU', 'concept_unit_EN', 'gram_structure_EN']].head())

Доступные колонки: ['case_iD', 'sent_ID', 'sent_text_RU', 'token_ID', 'token_RU', 'token_pos', 'concept_unit_RU', 'type_RU', 'connotation_RU', 'тональность_RU', 'gram_structure', 'comments_RU', 'sent_text_EN', 'concept_unit_EN', 'token_EN', 'token_pos_EN', 'gram.structure_EN', 'type_EN', 'connotation_EN', 'тональность_EN', 'comments_EN', 'translation_shift', 'cosine_sim_LaBSE', 'shift_notes', 'verified']
✅ Обработка завершена


#### Сохранение

In [19]:
# 2. Открываем оригинал через openpyxl
wb = load_workbook(FILENAME)
ws = wb['Разметка']

# 3. Находим колонки (один раз)
gram_col = None
gram_en_col = None
for col in range(1, ws.max_column + 1):
    val = ws.cell(2, col).value
    if val == 'gram_structure_RU':
        gram_col = col
    elif val == 'gram_structure_EN':
        gram_en_col = col

In [20]:
# 4. Записываем результаты (построчно, но быстро)
for idx, (_, row) in enumerate(df.iterrows()):
    excel_row = idx + 3
    if gram_col:
        ws.cell(excel_row, gram_col).value = row['gram_structure_RU']
    if gram_en_col:
        ws.cell(excel_row, gram_en_col).value = row['gram_structure_EN']

# 5. Сохраняем
wb.save(FILENAME)
print(f"✅ Готово! Обновлено {len(df)} строк")

✅ Готово! Обновлено 1014 строк


### Этап II - Косинусная близость

#### Косинусная близость

In [21]:
from sentence_transformers import SentenceTransformer

In [ ]:
df = pd.read_excel(FILENAME, sheet_name='Разметка', header=0)

In [23]:
model = SentenceTransformer('sentence-transformers/LaBSE')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2651.92it/s]


In [24]:
# ru_embeddings = model.encode(
#     df['concept_unit_RU'].fillna('').astype(str).tolist(),
#     batch_size=32,
#     show_progress_bar=True,
#     normalize_embeddings=True
# )

# print("Кодирование английских фраз...")
# en_embeddings = model.encode(
#     df['concept_unit_EN'].fillna('').astype(str).tolist(),
#     batch_size=32,
#     show_progress_bar=True,
#     normalize_embeddings=True
# )

In [25]:
ru_embeddings = model.encode(
    df['sent_text_RU'].fillna('').astype(str).tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Кодирование английских фраз...")
en_embeddings = model.encode(
    df['sent_text_EN'].fillna('').astype(str).tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 32/32 [01:28<00:00,  2.76s/it]


Кодирование английских фраз...


Batches: 100%|██████████| 32/32 [01:11<00:00,  2.24s/it]


In [26]:
# Вычисляем косиносное сходство
df['cosine_sim_LaBSE'] = (ru_embeddings * en_embeddings).sum(axis=1)
df


,case_iD,sent_ID,sent_text_RU,token_ID,token_RU,token_pos,concept_unit_RU,type_RU,connotation_RU,тональность_RU,...,token_pos_EN,gram.structure_EN,type_EN,connotation_EN,тональность_EN,comments_EN,translation_shift,cosine_sim_LaBSE,shift_notes,verified
0,NaN,NaN,За все годы лихорадочной работы в моргах ― с ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.849738,NaN,NaN
1,NaN,NaN,"Когда река успокоилась, кто-то переплыл на ос...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.878006,NaN,NaN
2,NaN,NaN,В палатах стоял тяжелый смрад – лежачие стару...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.833219,NaN,NaN
3,NaN,NaN,"Там, под троллейбусом, была тьма и жуткая тес...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.695375,NaN,NaN
4,NaN,NaN,Раздался легкий выстрел — и смрад на всю дере...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.684042,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1009,NaN,NaN,"На следующее утро, когда Гарри вышел к завтра...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.883550,NaN,NaN
1010,NaN,NaN,"Вошли Дудли с дядей Верноном, брезгливо морща...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.828051,NaN,NaN
1011,NaN,NaN,Вскоре хижина наполнилась запахом потрескивав...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.740455,NaN,NaN
1012,NaN,NaN,"Потом они посетили аптеку, где было достаточн...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.759488,NaN,NaN


#### Разметка translation_shift и shift_notes ##

In [27]:
def classify_shift(row):
    cos = row['cosine_sim_LaBSE']
    
    if cos >= THRESHOLD_EQUIV:
        return 'equivalent'
    elif cos >= THRESHOLD_SHIFT:
        return 'shift'
    else:
        return 'significant_shift'

df['translation_shift'] = df.apply(classify_shift, axis=1)
df 

,case_iD,sent_ID,sent_text_RU,token_ID,token_RU,token_pos,concept_unit_RU,type_RU,connotation_RU,тональность_RU,...,token_pos_EN,gram.structure_EN,type_EN,connotation_EN,тональность_EN,comments_EN,translation_shift,cosine_sim_LaBSE,shift_notes,verified
0,NaN,NaN,За все годы лихорадочной работы в моргах ― с ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,shift,0.849738,NaN,NaN
1,NaN,NaN,"Когда река успокоилась, кто-то переплыл на ос...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,shift,0.878006,NaN,NaN
2,NaN,NaN,В палатах стоял тяжелый смрад – лежачие стару...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,shift,0.833219,NaN,NaN
3,NaN,NaN,"Там, под троллейбусом, была тьма и жуткая тес...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,significant_shift,0.695375,NaN,NaN
4,NaN,NaN,Раздался легкий выстрел — и смрад на всю дере...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,significant_shift,0.684042,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1009,NaN,NaN,"На следующее утро, когда Гарри вышел к завтра...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,shift,0.883550,NaN,NaN
1010,NaN,NaN,"Вошли Дудли с дядей Верноном, брезгливо морща...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,shift,0.828051,NaN,NaN
1011,NaN,NaN,Вскоре хижина наполнилась запахом потрескивав...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,significant_shift,0.740455,NaN,NaN
1012,NaN,NaN,"Потом они посетили аптеку, где было достаточн...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,shift,0.759488,NaN,NaN


#### Сохранение

In [ ]:
#  Сохраняем результаты в отдельный файл (если надо)
df.to_excel('../results/tables/result_with_shifts.xlsx', index=False)
print("✅ Готово!")


✅ Готово!
